# [NOTE] ROTBOTS -- Subtitles

---

TikTok-style word-by-word subtitles. Skip if `ENABLE_SUBTITLES = False`.

---

In [ ]:
# SETUP
!pip install -q faster-whisper
import json, random, subprocess, time as _time
from pathlib import Path
from IPython.display import display, Markdown, HTML
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
def progress_bar(c,t,l='',w=30):
    p=c/t if t>0 else 0;f=int(w*p);return f'<div style="font-family:monospace;font-size:14px;padding:2px 0;">{"#"*f}{"-"*(w-f)} {c}/{t} {l} ({p:.0%})</div>'
def progress_html(title,c,t,l='',d='',color='#4ecca3'):
    return f'<div style="background:#1a1a2e;padding:12px;border-radius:8px;color:#eaeaea;font-family:monospace;"><div style="font-size:16px;font-weight:bold;color:{color};">{title}</div>{progress_bar(c,t,l)}'+(f'<div style="color:#a0a0a0;font-size:12px;margin-top:4px;">{d}</div>' if d else '')+'</div>'
print('[OK] Setup')

In [ ]:
# SESSION
sessions=sorted([d.name for d in BASE_DIR.iterdir() if d.is_dir() and (d/'video_plan.json').exists()])
SESSION_NAME=sessions[-1] if sessions else ''
SESSION_DIR=BASE_DIR/SESSION_NAME
with open(SESSION_DIR/'video_plan.json') as f: plan=json.load(f)
AUDIO_DIR=SESSION_DIR/'audio'
print(f'[OK] {SESSION_NAME}')

In [ ]:
# SETTINGS
SUBTITLE_MODE='random'
MIXUP_MIN=10;MIXUP_MAX=25
STYLES={
    '01':{'font':'Impact','size':80,'bold':-1,'primary':'&H00FFFFFF','outline_c':'&H00000000','back':'&H00000000','border_style':1,'outline':7,'shadow':0,'alignment':5,'margin_v':20,'upper':True,'anim':r'{\fscx0\fscy0\t(0,40,\fscx130\fscy130)\t(40,80,\fscx100\fscy100)}','desc':'Classic Impact'},
    '04':{'font':'Arial Black','size':76,'bold':-1,'primary':'&H00FFFFFF','outline_c':'&H008030FF','back':'&H00000000','border_style':1,'outline':8,'shadow':3,'alignment':5,'margin_v':30,'upper':True,'anim':r'{\fscx0\fscy0\t(0,30,\fscx140\fscy140)\t(30,70,\fscx100\fscy100)}','desc':'Purple bounce'},
    '05':{'font':'Arial Black','size':70,'bold':0,'primary':'&H00FFFFFF','outline_c':'&H00000000','back':'&HCC1A1A1A','border_style':3,'outline':16,'shadow':0,'alignment':2,'margin_v':80,'upper':True,'anim':r'{\fad(80,0)\fscx95\fscy95\t(0,60,\fscx100\fscy100)}','desc':'Dark box'},
    '06':{'font':'Impact','size':85,'bold':0,'primary':'&H0000DDFF','outline_c':'&H000000CC','back':'&H00000000','border_style':1,'outline':7,'shadow':4,'alignment':5,'margin_v':20,'upper':True,'anim':r'{\fscx0\fscy0\t(0,25,\fscx150\fscy150)\t(25,55,\fscx90\fscy90)\t(55,80,\fscx105\fscy105)\t(80,100,\fscx100\fscy100)}','desc':'Orange spring'},
    '08':{'font':'Arial Black','size':72,'bold':0,'primary':'&H00FFFF00','outline_c':'&H00FF0066','back':'&H00000000','border_style':1,'outline':5,'shadow':5,'alignment':5,'margin_v':20,'upper':True,'anim':r'{\fscx0\fscy0\t(0,40,\fscx115\fscy115)\t(40,80,\fscx100\fscy100)}','desc':'Neon pop-in'},
}
for k,v in STYLES.items(): print(f'   {k}: {v["desc"]}')

In [ ]:
# WHISPER
from faster_whisper import WhisperModel
prog = display(HTML(progress_html('Loading Whisper...', 0, 1)), display_id=True)
wm = WhisperModel('base', device='cpu', compute_type='int8')

narration_file = AUDIO_DIR / 'narration_full.mp3'
all_words = []
t0 = _time.time()

if narration_file.exists():
    prog.update(HTML(progress_html('Transcribing narration...', 0, 1, 'files')))
    segs, info = wm.transcribe(str(narration_file), word_timestamps=True, language='en')
    for seg in segs:
        if seg.words:
            for w in seg.words:
                all_words.append({'word': w.word.strip(), 'start': w.start, 'end': w.end})
    prog.update(HTML(progress_html(f'{len(all_words)} words ({_time.time()-t0:.1f}s)', 1, 1, 'files')))
else:
    print('No narration_full.mp3 found!')


In [ ]:
# GENERATE ASS
W=1280;H=720;sc=H/512
def sl(n,s):
    sz=int(s['size']*sc);ol=int(s['outline']*sc);mv=int(s['margin_v']*sc)
    return f'Style: {n},{s["font"]},{sz},{s["primary"]},&H0000FFFF,{s["outline_c"]},{s["back"]},{s["bold"]},0,0,0,100,100,2,0,{s["border_style"]},{ol},{s["shadow"]},{s["alignment"]},10,10,{mv},1'
def fmt(sec):
    h=int(sec//3600);m=int((sec%3600)//60);s=int(sec%60);cs=int((sec%1)*100)
    return f'{h}:{m:02d}:{s:02d}.{cs:02d}'
mx=SUBTITLE_MODE=='random'
sls=[sl(f'TT{k}',v) for k,v in STYLES.items()] if mx else [sl('TT',STYLES.get(SUBTITLE_MODE,STYLES['01']))]
hdr=f'[Script Info]\nTitle: ROTBOTS\nScriptType: v4.00+\nWrapStyle: 0\nScaledBorderAndShadow: yes\nPlayResX: {W}\nPlayResY: {H}\n\n[V4+ Styles]\nFormat: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding\n'+'\n'.join(sls)+'\n\n[Events]\nFormat: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text\n'
keys=list(STYLES.keys());ck=random.choice(keys) if mx else SUBTITLE_MODE if SUBTITLE_MODE in STYLES else '01'
ns=random.uniform(MIXUP_MIN,MIXUP_MAX) if mx else 99999
dlg=[]
for w in all_words:
    if mx and w['start']>=ns: ck=random.choice(keys);ns=w['start']+random.uniform(MIXUP_MIN,MIXUP_MAX)
    sty=STYLES[ck];sn=f'TT{ck}' if mx else 'TT'
    wd=w['word'].replace('\\','\\\\').replace('{','\\{').replace('}','\\}')
    dlg.append(f'Dialogue: 0,{fmt(w["start"])},{fmt(w["end"])},{sn},,0,0,0,,{sty["anim"]}{wd.upper() if sty["upper"] else wd}')
with open(SESSION_DIR/'subtitles.ass','w',encoding='utf-8') as f: f.write(hdr+'\n'.join(dlg))
print(f'[OK] subtitles.ass: {len(dlg)} words')

---
*ROTBOTS Workshop -- LI-MA 2026*